# Bronze Layer: Ingestion & Extraction Notebook
Extracts source data from the local database via the Bore TCP tunnel into Databricks using PyMySQL query pushdown, secure secret credentials, and centralized task logging.

In [ ]:
# 1. Install Driver Library for Serverless / Community Compute
%pip install pymysql cryptography --quiet

In [ ]:
%run ../../src/utilities/logger

In [ ]:
# 3. Initialize SparkSession and Task Logger
from pyspark.sql import SparkSession
import pymysql
import pandas as pd

# Ensure SparkSession is active
spark = SparkSession.builder.getOrCreate()

# Initialize logger for this notebook execution
logger = get_task_logger(notebook_name_override="bronze_extraction")
logger.info("=== Starting Bronze Extraction Notebook Execution ===")

In [ ]:
# 4. Interactive Widget for Dynamic Tunnel Port
# Set default to current active tunnel port
dbutils.widgets.text("tunnel_port", "17546", "Bore Tunnel Port")
db_port = dbutils.widgets.get("tunnel_port")

logger.info(f"Using Bore Tunnel Port: {db_port}")
print(f"Active Tunnel Port: {db_port}")

In [ ]:
# 5. Retrieve Credentials Securely from Databricks Secrets
SECRET_SCOPE = "wanderbricks_scope"

db_user = dbutils.secrets.get(scope=SECRET_SCOPE, key="mysql_user")
db_password = dbutils.secrets.get(scope=SECRET_SCOPE, key="mysql_password")
db_host = dbutils.secrets.get(scope=SECRET_SCOPE, key="tunnel_host")
db_name = dbutils.secrets.get(scope=SECRET_SCOPE, key="mysql_db")

logger.info(f"Retrieved credentials from scope '{SECRET_SCOPE}' for user '{db_user}'")
print(f"Connecting to Database: {db_name} at Host: {db_host} as User: {db_user}")

In [ ]:
# 6. Query Pushdown & Extraction into PySpark DataFrame
table_to_extract = "countries"
pushdown_query = f"SELECT country, country_code, continent FROM {table_to_extract}"

logger.info(f"Executing pushdown query on MySQL: {pushdown_query}")

# Connect to MySQL over Bore TCP tunnel with a 10s connect timeout
connection = pymysql.connect(
    host=db_host,
    port=int(db_port),
    user=db_user,
    password=db_password,
    database=db_name,
    connect_timeout=10
)

try:
    # Query pushed down to MySQL engine
    pdf = pd.read_sql_query(pushdown_query, con=connection)
finally:
    connection.close()

# Convert directly to PySpark DataFrame
df = spark.createDataFrame(pdf)

record_count = df.count()
logger.info(f"Successfully extracted {record_count} rows from table '{table_to_extract}'")
print(f"Extracted {record_count} rows from {table_to_extract}:")

display(df)